# IE212 – Lab 3: Spark DataFrame
## Notebook thực hành hoàn chỉnh

Chạy lần lượt bằng **Runtime → Run all** trên Google Colab. Khi được hỏi, chọn 5 file MovieLens hoặc `data.zip`. Dữ liệu đủ trong runtime sẽ được dùng lại.

Dựa trên notebook gốc và hướng dẫn thực hành của giảng viên. Các giá trị trong comment là mốc đối chiếu từ tài liệu; bảng output được tạo khi chạy Spark.


---
# PHẦN A. CHUẨN BỊ TRƯỚC BUỔI HỌC

## A.1. Môi trường thực hành: Google Colab

### Bước 1. Cài PySpark và tạo SparkSession

Dùng PySpark 3.5.1 và Java 17. Cell tiếp theo tự chọn hoặc cài Java 17 trong runtime Colab.


In [1]:
%pip install -q pyspark==3.5.1


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
from google.colab import files
import zipfile
import io
import os

# Upload data.zip neu chua co du lieu
if not os.path.exists("/content/data/movielens-mini/movies.txt") or \
   not os.path.exists("/content/data/fecom/Orders.csv"):

    uploaded = files.upload()

    with zipfile.ZipFile(io.BytesIO(uploaded["data.zip"])) as z:
        z.extractall("/content/")

# Kiem tra du lieu
print("MovieLens:", os.listdir("/content/data/movielens-mini"))
print("Fecom:", os.listdir("/content/data/fecom"))

ML = "/content/data/movielens-mini"
FE = "/content/data/fecom"

print("Duong dan MovieLens:", ML)
print("Duong dan Fecom:", FE)

Saving data.zip to data.zip
MovieLens: ['ratings_2.txt', 'ratings_1.txt', 'occupation.txt', 'movies.txt', 'users.txt']
Fecom: ['Customer_List.csv', 'Order_Items.csv', 'Order_Reviews.csv', 'Products.csv', 'Orders.csv']
Duong dan MovieLens: /content/data/movielens-mini
Duong dan Fecom: /content/data/fecom


In [3]:
import os
import subprocess
from pathlib import Path

# Spark 3.5.1 dùng Java 17 trên Colab.
if os.name == "posix":
    java17 = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
    if not java17:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"], check=True)
        java17 = sorted(Path("/usr/lib/jvm").glob("java-17-openjdk-*"))
    if not java17:
        raise RuntimeError("Không tìm thấy Java 17 sau khi cài đặt.")
    os.environ["JAVA_HOME"] = str(java17[0])

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("IE212-Lab3")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("PySpark:", spark.version)
assert spark.version == "3.5.1", "Hãy dùng runtime mới rồi chạy lại từ cell cài đặt."


PySpark: 3.5.1


### Bước 2. Load dữ liệu của lab lên Colab

Chọn đủ 5 file `.txt` hoặc một ZIP chứa chúng. Cell tự nhận diện cả ZIP có thư mục `data/`, `movielens-mini/` hoặc thư mục cha khác; chỉ đưa các file cần dùng vào đúng đường dẫn. Nếu dữ liệu đã đủ thì không mở cửa sổ upload.


In [4]:
import os
from pathlib import Path, PurePosixPath
from io import BytesIO
from zipfile import ZipFile, BadZipFile


def prepare_data(directory, required):
    target = Path(directory)
    target.mkdir(parents=True, exist_ok=True)
    missing = [name for name in required
               if not (target / name).is_file() or (target / name).stat().st_size == 0]
    if missing:
        print("Còn thiếu:", ", ".join(missing))
        print("Chọn các file dữ liệu hoặc data.zip trong cửa sổ upload.")
        try:
            from google.colab import files
        except ImportError as exc:
            raise RuntimeError("Hãy chạy notebook trên Google Colab để upload dữ liệu.") from exc
        uploaded = files.upload()
        expected = {name.casefold(): name for name in required}
        found = {}

        def remember(name, payload):
            filename = PurePosixPath(name.replace("\\", "/")).name
            canonical = expected.get(filename.casefold())
            if canonical:
                if canonical in found and found[canonical] != payload:
                    raise ValueError(f"Có nhiều bản khác nội dung của {canonical} trong lần upload.")
                found[canonical] = payload

        for name, payload in uploaded.items():
            if name.lower().endswith(".zip"):
                try:
                    with ZipFile(BytesIO(payload)) as archive:
                        for member in archive.infolist():
                            parts = PurePosixPath(member.filename.replace("\\", "/")).parts
                            if member.is_dir() or "__MACOSX" in parts:
                                continue
                            if parts and parts[-1].casefold() in expected:
                                remember(member.filename, archive.read(member))
                except BadZipFile as exc:
                    raise ValueError(f"ZIP không hợp lệ: {name}") from exc
            else:
                remember(name, payload)
        # Chỉ ghi đúng tên file cần dùng; không giữ cấu trúc lồng trong ZIP.
        for name, payload in found.items():
            (target / name).write_bytes(payload)
    present = [name for name in required
               if (target / name).is_file() and (target / name).stat().st_size > 0]
    print("Tìm thấy:", ", ".join(present))
    missing = sorted(set(required) - set(present))
    if missing:
        raise FileNotFoundError("Còn thiếu: " + ", ".join(missing)
                                + ". Chạy lại cell và chọn bổ sung các file này.")
    print("Thư mục dữ liệu:", target)
    return target

ML = "/content/data/movielens-mini"
ML_FILES = ("movies.txt", "ratings_1.txt", "ratings_2.txt", "users.txt", "occupation.txt")
prepare_data(ML, ML_FILES)


Tìm thấy: movies.txt, ratings_1.txt, ratings_2.txt, users.txt, occupation.txt
Thư mục dữ liệu: /content/data/movielens-mini


PosixPath('/content/data/movielens-mini')

## A.2. Dữ liệu của lab

Năm tệp MovieLens (bốn bảng), phân tách bằng dấu phẩy, không có dòng tiêu đề. Đúng bộ của Lab 2:

```
movies.txt      : MovieID, Title, Genres          (Genres phân tách bằng "|")
ratings_1/2.txt : UserID, MovieID, Rating, Timestamp
users.txt       : UserID, Gender, Age, Occupation, Zip-code
occupation.txt  : ID, Occupation
```


In [5]:
# Kiểm tra lại trước khi đọc các bảng.
assert all((Path(ML) / name).is_file() for name in ML_FILES)
print(sorted(os.listdir(ML)))


['movies.txt', 'occupation.txt', 'ratings_1.txt', 'ratings_2.txt', 'users.txt']


### Dựng lại RDD `ratings` của Lab 2

Phần Làm quen ở dưới cần RDD `ratings` của Lab 2 để minh hoạ `toDF()`. Ô dưới dựng
lại RDD đó trong một lần, giống Bước 3 và Bước 4 của Lab 2 B.1.

In [6]:
def parse_rating(line):
    uid, mid, rating, ts = line.split(",")
    return (int(uid), int(mid), float(rating), int(ts))


ratings = sc.textFile(ML + "/ratings_1.txt") \
            .union(sc.textFile(ML + "/ratings_2.txt")) \
            .map(parse_rating).cache()

print(ratings.count())      # 184, khớp với Lab 2

184


---
# PHẦN B. LÀM QUEN VỚI SPARK DATAFRAME

DataFrame là một tập hợp phân tán các dữ liệu được gom nhóm theo các cột được đặt
tên, giống một bảng trong cơ sở dữ liệu quan hệ hoặc một bảng tính có dòng tiêu đề.
Mỗi cột có tên, kiểu dữ liệu và thuộc tính nullable; cả ba thứ đó hợp thành lược đồ
(schema) của DataFrame.

Khác biệt cốt lõi so với RDD: RDD là một tập hợp các đối tượng Python và Spark không
biết bên trong mỗi phần tử có gì. Với Spark, `(1001, ('The Godfather (1972)', 'Crime|Drama'))`
chỉ là một đối tượng. DataFrame thì có lược đồ, nên Spark biết tên và kiểu của từng
cột. Nhờ đó Spark chạy được bộ tối ưu: nó đọc cả chuỗi phép tính trước khi chạy, bỏ
những cột không dùng tới, đổi thứ tự các phép lọc, rồi sinh mã Java chạy thẳng trên
JVM. Với RDD trong PySpark, mọi lambda của bạn phải chạy trong một tiến trình Python
riêng và từng bản ghi phải chuyển qua lại giữa JVM và Python.

## Làm quen: tạo DataFrame từ RDD và từ danh sách Python

DataFrame tạo được từ nhiều nguồn: từ RDD bằng `toDF()` hoặc `createDataFrame()`,
từ một danh sách Python, hoặc từ tệp dữ liệu như CSV, TXT, JSON, Parquet.

Cách nhanh nhất để thấy khác biệt với RDD: đổi RDD `ratings` vừa dựng sang DataFrame
rồi xem lược đồ của nó.

In [7]:
# toDF() đổi một RDD các tuple thành DataFrame, tham số là danh sách tên cột.
df_tu_rdd = ratings.toDF(["UserID", "MovieID", "Rating", "Timestamp"])

df_tu_rdd.printSchema()
df_tu_rdd.show(3)

# Spark suy ra kiểu từ chính dữ liệu: số nguyên của Python thành long, số thực
# thành double. Ở B.1 lược đồ được khai báo tường minh nên MovieID sẽ là int.

root
 |-- UserID: long (nullable = true)
 |-- MovieID: long (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Timestamp: long (nullable = true)

+------+-------+------+----------+
|UserID|MovieID|Rating| Timestamp|
+------+-------+------+----------+
|     7|   1020|   4.5|1577836800|
|    23|   1015|   3.5|1577923200|
|    45|   1030|   4.0|1578009600|
+------+-------+------+----------+
only showing top 3 rows



In [8]:
# Tạo thẳng từ một danh sách Python, lược đồ viết bằng chuỗi DDL.
df_vi_du = spark.createDataFrame(
    [(1, "Nam", 20), (2, "Lan", 22), (3, "Binh", 19)],
    "id INT, ten STRING, tuoi INT")

df_vi_du.show()
print(df_vi_du.columns)         # ['id', 'ten', 'tuoi']

+---+----+----+
| id| ten|tuoi|
+---+----+----+
|  1| Nam|  20|
|  2| Lan|  22|
|  3|Binh|  19|
+---+----+----+

['id', 'ten', 'tuoi']


Bảng tra cứu khi chuyển từ cách nghĩ RDD sang DataFrame:

| Phép trên RDD | Phép DataFrame tương đương |
|---|---|
| `map(f)` | `select()`, `withColumn()` |
| `filter(f)` | `filter()`, `where()` |
| `flatMap` tách một chuỗi | `explode(split(...))` |
| `reduceByKey(f)` | `groupBy(...).agg(...)` |
| `mapValues(f)` | `withColumn()` |
| `join(other)` | `join(other, on=..., how=...)` |
| `distinct()` | `distinct()`, `dropDuplicates()` |
| `sortBy(f)` rồi `collect()` + `sorted()` | `sort()`, `orderBy()` |
| `take(n)` | `show(n)`, `take(n)` |
| không có phép tương đương | các hàm Window: `row_number()`, `rank()` |

Từ đây trở đi mọi phép đều làm trên DataFrame. Đừng quay về RDD bằng `.rdd` rồi dùng
lại lambda cũ, làm vậy là mất phần tối ưu vừa nói ở trên.

## B.1. DataFrame và lược đồ

### Mục tiêu

- Tạo được DataFrame từ tệp CSV không có dòng tiêu đề, bằng lược đồ tường minh.
- Đọc được `printSchema()` và biết kiểm tra Spark hiểu từng cột là kiểu gì.
- Biết hai cách khai báo lược đồ: chuỗi DDL và đối tượng `StructType`.

### Khái niệm cần nắm

Khi đọc CSV, ba tuỳ chọn hay dùng nhất là `header` để lấy tên cột từ dòng đầu,
`delimiter` để khai báo ký tự tách cột, và `inferSchema` để Spark tự đoán kiểu dữ liệu.

Lược đồ khai báo được bằng hai cách. Cách thứ nhất là một chuỗi theo định dạng DDL,
ngắn gọn, phù hợp với lược đồ phẳng. Cách thứ hai là đối tượng `StructType`, dài hơn
nhưng khai báo được cả lược đồ lồng nhau. Lab này dùng chuỗi DDL.

### Bước 1. Tạo bốn DataFrame bằng chuỗi DDL

Bốn tệp MovieLens không có dòng tiêu đề, nên không truyền `header` và phải tự khai
báo tên cột.

In [9]:
df_movies = spark.read.schema(
    "MovieID INT, Title STRING, Genres STRING").option("header", False).csv(ML + "/movies.txt")

# Một lệnh read.csv nhận được danh sách nhiều đường dẫn, nên không cần union như bên RDD.
df_ratings = spark.read.schema(
    "UserID INT, MovieID INT, Rating DOUBLE, Timestamp BIGINT") \
    .option("header", False).csv([ML + "/ratings_1.txt", ML + "/ratings_2.txt"])

df_users = spark.read.schema(
    "UserID INT, Gender STRING, Age INT, Occupation INT, Zip STRING") \
    .option("header", False).csv(ML + "/users.txt")

df_occ = spark.read.schema(
    "OccID INT, Occupation STRING").option("header", False).csv(ML + "/occupation.txt")

df_ratings.cache()      # df_ratings được dùng lại ở hầu hết các mục sau

df_movies.printSchema()
print(df_movies.count(), df_ratings.count(), df_users.count(), df_occ.count())
# 50 184 50 15
ml_counts = [("movies", df_movies.count(), 50),
             ("ratings", df_ratings.count(), 184),
             ("users", df_users.count(), 50),
             ("occupation", df_occ.count(), 15)]
spark.createDataFrame(ml_counts, "Bang STRING, So_dong LONG, Moc_doi_chieu LONG").show()
assert all(actual == expected for _, actual, expected in ml_counts), "Kiểm tra lại file MovieLens."


root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

50 184 50 15
+----------+-------+-------------+
|      Bang|So_dong|Moc_doi_chieu|
+----------+-------+-------------+
|    movies|     50|           50|
|   ratings|    184|          184|
|     users|     50|           50|
|occupation|     15|           15|
+----------+-------+-------------+



### Bước 2. Xem dữ liệu bằng `show()`

In [10]:
df_movies.show(5, truncate=40)

# truncate cắt phần hiển thị của các ô dài rồi thay bằng ba dấu chấm. Dữ liệu không
# mất, chỉ hiển thị bị cắt. Muốn xem đủ thì dùng show(5, truncate=False).

+-------+-------------------------------+-----------------------+
|MovieID|                          Title|                 Genres|
+-------+-------------------------------+-----------------------+
|   1001|           The Godfather (1972)|            Crime|Drama|
|   1002|The Shawshank Redemption (1994)|                  Drama|
|   1003|        Schindler's List (1993)|Biography|Drama|History|
|   1004|             Raging Bull (1980)|  Biography|Drama|Sport|
|   1005|              Casablanca (1942)|      Drama|Romance|War|
+-------+-------------------------------+-----------------------+
only showing top 5 rows



### Bước 3. Lỗi cố ý: dùng `header=True` với tệp không có tiêu đề

In [11]:
sai = spark.read.csv(ML + "/ratings_1.txt", header=True, inferSchema=True)
print(sai.count())      # 83
print(sai.columns)      # ['7', '1020', '4.5', '1577836800']

# ratings_1.txt có 84 dòng dữ liệu. header=True lấy dòng đầu làm tiêu đề, nên chỉ
# còn 83 dòng và tên bốn cột trở thành chính giá trị của bản ghi đầu tiên.
# Lỗi này không làm chương trình dừng, nó chỉ âm thầm mất một bản ghi và đặt tên
# cột vô nghĩa. Luôn chạy printSchema() ngay sau khi đọc.

83
['7', '1020', '4.5', '1577836800']


### Bước 4. Cùng lược đồ đó viết bằng `StructType`

Ô dưới khai báo lại lược đồ của `movies.txt` bằng `StructType` và phải ra lược đồ
giống hệt bản chuỗi DDL ở Bước 1. Dạng `StructType` dài hơn nhưng khai báo được lược
đồ lồng nhau và đặt được thuộc tính nullable cho từng cột.

In [12]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_movies = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title",   StringType(),  True),
    StructField("Genres",  StringType(),  True),
])

df_movies_2 = spark.read.schema(schema_movies).csv(ML + "/movies.txt")
print(df_movies_2.schema == df_movies.schema)    # True

# Khai báo lược đồ tường minh còn nhanh hơn inferSchema: inferSchema buộc Spark đọc
# thêm một lượt toàn bộ dữ liệu chỉ để đoán kiểu.
#
# Đọc movies.txt thành 3 cột là an toàn vì bộ này không có tên phim nào chứa dấu
# phẩy. Nếu có, tệp CSV buộc phải bọc tên phim trong dấu nháy kép. Đây là lý do ở
# Lab 2 phải viết split(",", 2) thay vì split(",").

True


### Câu hỏi kiểm tra nhanh

1. Vì sao `printSchema()` của `df_movies` ghi `nullable = true` cho cả ba cột, dù tệp
   không có ô nào rỗng?
2. Nếu khai báo `Rating` là `INT` thay vì `DOUBLE` thì cột đó đọc ra giá trị gì?
**Trả lời:** `nullable=True` là thuộc tính lược đồ, không khẳng định dữ liệu thực tế có NULL. Với CSV và chế độ đọc mặc định, chuỗi điểm dạng `4.5` không đọc được thành INT sẽ thành NULL; dùng DOUBLE để giữ phần thập phân.


## B.2. Các hàm cơ bản trên DataFrame

### Mục tiêu

- Dùng được `select`, `withColumn`, `filter`, `distinct`, `dropDuplicates` và `sort`.
- Biết các hàm thông dụng trong `pyspark.sql.functions` và vì sao phải import module đó.
- Nhận ra hàm `split` của DataFrame nhận biểu thức chính quy, không phải chuỗi thường.

### Khái niệm cần nắm

Nhóm hàm thuộc đối tượng DataFrame gồm: `select` chọn cột, `withColumn` tạo hoặc sửa
một cột, `filter` lọc dòng, `distinct` bỏ dòng trùng xét toàn bộ cột, `dropDuplicates`
bỏ dòng trùng xét một số cột được chỉ định, và `sort` sắp xếp theo một hoặc nhiều cột.

`withColumn` dùng để đổi kiểu, cập nhật giá trị, tạo cột từ cột có sẵn hoặc tạo cột hằng.
Đổi tên cột dùng `withColumnRenamed()`.

Phần lớn các hàm dùng để tính toán trên cột không nằm trong đối tượng DataFrame mà nằm
trong module `pyspark.sql.functions`, nên phải import trước khi dùng. Module này gồm
các nhóm: hàm thông thường (`col`, `lit`, `when`, `coalesce`, `isnull`), hàm toán học,
hàm ngày tháng (`year`, `month`, `to_timestamp`, `date_format`), hàm về collection
(`split`, `size`, `explode`, `array_contains`), hàm tổng hợp dữ liệu (`avg`, `count`,
`collect_list`), hàm Window (`row_number`, `rank`, `lag`) và hàm chuỗi.

### Bước 1. `select` và `filter`


In [13]:
from pyspark.sql.functions import col, split, size

df_ratings.select("MovieID", "Rating").show(5)
print(df_ratings.filter((col("Rating") >= 4.0) & (col("MovieID") == 1020)).count())
# 6

# Mỗi điều kiện trong filter phải bọc trong một cặp ngoặc riêng, và dùng & thay cho
# and, | thay cho or. Viết col("Rating") >= 4.0 and col("MovieID") == 1020 sẽ báo
# lỗi, vì Python không gọi được toán tử and trên đối tượng Column.
# where là cách viết khác của filter.
df_ratings.where((col("Rating") >= 4.0) & (col("MovieID") == 1020)).show()


+-------+------+
|MovieID|Rating|
+-------+------+
|   1012|   3.5|
|   1039|   4.0|
|   1043|   4.5|
|   1020|   3.0|
|   1050|   4.0|
+-------+------+
only showing top 5 rows

6
+------+-------+------+----------+
|UserID|MovieID|Rating| Timestamp|
+------+-------+------+----------+
|    22|   1020|   4.5|1578874600|
|    37|   1020|   4.0|1580432000|
|    11|   1020|   4.5|1581987200|
|    26|   1020|   4.0|1583542400|
|     7|   1020|   4.5|1585097600|
|     7|   1020|   4.5|1577836800|
+------+-------+------+----------+



### Bước 2. `withColumn` tạo cột mới, và lỗi cố ý với `split`

Đếm số thể loại của mỗi phim bằng cách tách cột `Genres` rồi lấy kích thước mảng.
Chạy bản sai trước:

In [14]:
df_movies.withColumn("so_the_loai", size(split(col("Genres"), "|"))) \
         .show(5, truncate=40)

# The Godfather (1972) chỉ có hai thể loại mà ra 12.
# Ở Bước 1 của Lab 2 D.2 bạn viết split("|") trong Python và nó chạy đúng. Ở đây
# thì không: hàm split của pyspark.sql.functions nhận một biểu thức chính quy, giống
# Java ở Lab 1. Mẫu "|" trong biểu thức chính quy nghĩa là "rỗng hoặc rỗng", nên
# Spark tách ra từng ký tự: Crime|Drama có 11 ký tự, cộng thêm một chuỗi rỗng ở
# cuối mảng thành 12. Chương trình không báo lỗi.

+-------+-------------------------------+-----------------------+-----------+
|MovieID|                          Title|                 Genres|so_the_loai|
+-------+-------------------------------+-----------------------+-----------+
|   1001|           The Godfather (1972)|            Crime|Drama|         12|
|   1002|The Shawshank Redemption (1994)|                  Drama|          6|
|   1003|        Schindler's List (1993)|Biography|Drama|History|         24|
|   1004|             Raging Bull (1980)|  Biography|Drama|Sport|         22|
|   1005|              Casablanca (1942)|      Drama|Romance|War|         18|
+-------+-------------------------------+-----------------------+-----------+
only showing top 5 rows



Bản đúng phải chắn dấu gạch dọc bằng hai dấu chéo ngược:

In [15]:
df_movies.withColumn("so_the_loai", size(split(col("Genres"), "\\|"))) \
         .show(5, truncate=40)
# cột so_the_loai: 2, 1, 3, 3, 3
from pyspark.sql.functions import lit, when, array_contains, concat_ws, element_at, substring

# Đổi kiểu, cập nhật cột, tạo cột từ cột có sẵn và tạo cột hằng.
(df_ratings.withColumn("Timestamp", col("Timestamp").cast("long"))
 .withColumn("Rating", col("Rating") * lit(2.0))
 .withColumn("diem_cao", when(col("Rating") >= 8.0, "Cao").otherwise("Còn lại"))
 .withColumn("nguon", lit("MovieLens"))
 .withColumnRenamed("Rating", "Diem_thang_10")
 .select("UserID", "MovieID", "Diem_thang_10", "diem_cao", "nguon").show(5))

(df_movies.withColumn("genres_array", split(col("Genres"), r"\|"))
 .select("MovieID", array_contains("genres_array", "Drama").alias("co_Drama"),
         element_at("genres_array", 1).alias("the_loai_dau"),
         concat_ws(", ", "genres_array").alias("the_loai"),
         substring("Title", 1, 20).alias("ten_ngan"))
 .orderBy("MovieID").show(5, truncate=False))


+-------+-------------------------------+-----------------------+-----------+
|MovieID|                          Title|                 Genres|so_the_loai|
+-------+-------------------------------+-----------------------+-----------+
|   1001|           The Godfather (1972)|            Crime|Drama|          2|
|   1002|The Shawshank Redemption (1994)|                  Drama|          1|
|   1003|        Schindler's List (1993)|Biography|Drama|History|          3|
|   1004|             Raging Bull (1980)|  Biography|Drama|Sport|          3|
|   1005|              Casablanca (1942)|      Drama|Romance|War|          3|
+-------+-------------------------------+-----------------------+-----------+
only showing top 5 rows

+------+-------+-------------+--------+---------+
|UserID|MovieID|Diem_thang_10|diem_cao|    nguon|
+------+-------+-------------+--------+---------+
|    12|   1012|          7.0| Còn lại|MovieLens|
|    34|   1039|          8.0|     Cao|MovieLens|
|    27|   1043|       

### Bước 3. `distinct` và `dropDuplicates`

In [16]:
print(df_ratings.select("UserID").distinct().count())    # 49
print(df_ratings.select("MovieID").distinct().count())   # 14

df_ratings.select("Rating").distinct().sort("Rating").show()
# 3.0, 3.5, 4.0, 4.5

# Hai con số 49 và 14 giải thích luôn hai điều đã gặp ở Lab 2: users.txt có 50 dòng
# nhưng chỉ 49 người từng đánh giá, và movies.txt có 50 phim nhưng chỉ 14 phim được
# đánh giá. Bộ dữ liệu không có mức điểm 1.0, 1.5, 2.0 và 2.5.
print("Bỏ dòng trùng toàn bộ cột:", df_ratings.distinct().count())
print("Giữ một dòng mỗi UserID:", df_ratings.dropDuplicates(["UserID"]).count())
# Dòng được giữ bởi dropDuplicates không có thứ tự ưu tiên; ở đây chỉ dùng để đếm.


49
14
+------+
|Rating|
+------+
|   3.0|
|   3.5|
|   4.0|
|   4.5|
+------+

Bỏ dòng trùng toàn bộ cột: 184
Giữ một dòng mỗi UserID: 49


### Bước 4. `sort` theo nhiều cột

In [17]:
df_users.sort(col("Age").desc(), col("UserID")).show(5)
# UserID 29 (61 tuổi), 27 (54), 9 (50), 45 (50), 37 (48)

# Người dùng 9 và 45 cùng 50 tuổi. Khoá sắp xếp thứ hai là UserID, nên thứ tự giữa
# hai dòng bằng nhau trở thành xác định. Không có khoá thứ hai thì mỗi lần chạy có
# thể ra thứ tự khác. sort và orderBy là hai tên gọi của cùng một phép.

+------+------+---+----------+-----+
|UserID|Gender|Age|Occupation|  Zip|
+------+------+---+----------+-----+
|    29|     M| 61|         7|90123|
|    27|     M| 54|        15|78901|
|     9|     M| 50|         4|90123|
|    45|     M| 50|        12|56789|
|    37|     M| 48|         8|88990|
+------+------+---+----------+-----+
only showing top 5 rows



### Câu hỏi kiểm tra nhanh

1. `distinct()` và `dropDuplicates(["UserID"])` khác nhau ở điểm nào khi áp lên `df_ratings`?
2. Vì sao hàm `split` của DataFrame cần chắn ký tự `|` còn `str.split` của Python thì không?
**Trả lời:** `distinct()` xét tất cả cột; `dropDuplicates(["UserID"])` chỉ xét mã người dùng. Spark `split` nhận regex, trong đó `|` có ý nghĩa đặc biệt; Python `str.split` nhận chuỗi phân tách thường.


## B.3. Tổng hợp dữ liệu với `groupBy` và `agg`

### Mục tiêu

- Gộp theo khoá bằng `groupBy` và thực hiện nhiều phép tổng hợp cùng lúc bằng `agg`.
- Dùng `alias` để đặt tên cột kết quả.
- Giải lại Bài 1 bằng DataFrame và đối chiếu ra đúng 14 dòng của Lab 2 D.1.

### Khái niệm cần nắm

`groupBy` trả về một đối tượng `GroupedData`, trên đó gọi được các hàm tổng hợp sẵn
có: `count`, `mean`, `max`, `min`, `sum`, `avg`, `pivot`, và `agg`.

`agg` cho phép thực hiện nhiều phép tổng hợp cùng lúc. Bên trong `agg` dùng được nhóm
hàm gọi là agg_funcs của module `pyspark.sql.functions`, gồm `avg`, `count`,
`countDistinct`, `min`, `max`, `sum`, `first`, `last`, `stddev`, `collect_list`,
`collect_set` và nhiều hàm khác. Thông thường `alias` được dùng kèm để đổi tên cột
kết quả.

Khác với RDD, DataFrame có sẵn phép tính trung bình theo khoá. Cả đoạn `map` rồi
`reduceByKey` rồi `mapValues` của Lab 2 D.1 thu lại thành một lệnh `agg`.

### Bước 1. Bốn phép tổng hợp trong một lệnh `agg`

In [18]:
# min, max, round của Python trùng tên với hàm của Spark nên đổi tên khi import.
from pyspark.sql.functions import avg, count, min as smin, max as smax, round as sround

df_ratings.groupBy("MovieID").agg(
        count("*").alias("so_luot"),
        sround(avg("Rating"), 2).alias("diem_tb"),
        smin("Rating").alias("thap_nhat"),
        smax("Rating").alias("cao_nhat")
    ).sort("MovieID").show()
# 14 dòng, từ 1010 tới 1050

# Luôn gọi sort() trước show() khi xem kết quả của groupBy. Thứ tự dòng sau một phép
# gộp là không xác định vì nó phụ thuộc cách chia partition.
#
# Phim 1028 có thấp nhất và cao nhất đều là 3.5: cả 7 người đánh giá phim đó đều cho
# đúng 3.5 điểm. Cột diem_tb in ra 4.0 và 3.5 chứ không phải 4.00 và 3.50, vì round()
# trả về kiểu double và Spark in double theo cách ngắn nhất. Muốn đúng hai chữ số thì
# dùng format_number, nhưng hàm đó trả về chuỗi nên nếu sắp xếp theo cột đó sẽ sắp
# theo thứ tự chữ cái. Làm tròn để tính, định dạng chỉ ở bước in.

+-------+-------+-------+---------+--------+
|MovieID|so_luot|diem_tb|thap_nhat|cao_nhat|
+-------+-------+-------+---------+--------+
|   1010|     18|   3.44|      3.0|     4.5|
|   1012|      2|    4.0|      3.5|     4.5|
|   1013|     17|    4.0|      3.0|     4.5|
|   1015|      7|   4.36|      3.5|     4.5|
|   1020|     18|   3.67|      3.0|     4.5|
|   1025|     18|   4.06|      3.0|     4.5|
|   1028|      7|    3.5|      3.5|     3.5|
|   1030|      7|   3.14|      3.0|     4.0|
|   1037|     18|   3.89|      3.0|     4.5|
|   1039|     11|   3.82|      3.0|     4.5|
|   1040|     18|   3.61|      3.0|     4.5|
|   1043|     18|   3.89|      3.0|     4.5|
|   1047|      7|   3.86|      3.0|     4.0|
|   1050|     18|   3.47|      3.0|     4.5|
+-------+-------+-------+---------+--------+



### Bước 2. `collect_set`: việc mà Lab 1 phải tự viết Reducer

Ở Lab 1, bài TransAnalysis3 và TransAnalysis4 phải tự viết logic khử trùng lặp trong
Reducer để đếm số người khác nhau. Trong DataFrame đó là một hàm.

In [19]:
from pyspark.sql.functions import collect_set, countDistinct

df_ratings.groupBy("MovieID").agg(
        size(collect_set("UserID")).alias("so_nguoi_khac_nhau"),
        count("*").alias("so_luot")
    ).sort("MovieID").show()

# Hai cột lệch nhau ở phần lớn các phim, nghĩa là có người đánh giá cùng một phim
# nhiều lần. Phim 1050 lệch nhiều nhất: 18 lượt nhưng chỉ của 13 người.
# Cộng cột so_luot phải ra đúng 184.
from pyspark.sql.functions import sum as ssum

df_ratings.groupBy("MovieID").agg(count("*").alias("so_luot")) \
    .agg(ssum("so_luot").alias("tong_luot")).show()


+-------+------------------+-------+
|MovieID|so_nguoi_khac_nhau|so_luot|
+-------+------------------+-------+
|   1010|                16|     18|
|   1012|                 2|      2|
|   1013|                16|     17|
|   1015|                 7|      7|
|   1020|                17|     18|
|   1025|                16|     18|
|   1028|                 6|      7|
|   1030|                 5|      7|
|   1037|                16|     18|
|   1039|                11|     11|
|   1040|                16|     18|
|   1043|                17|     18|
|   1047|                 7|      7|
|   1050|                13|     18|
+-------+------------------+-------+

+---------+
|tong_luot|
+---------+
|      184|
+---------+



In [20]:
# countDistinct cho ra cùng con số mà không phải gom danh sách, nên rẻ hơn.
# collect_set chỉ cần khi bạn muốn giữ lại chính danh sách đó.
df_ratings.groupBy("MovieID").agg(
        countDistinct("UserID").alias("so_nguoi_khac_nhau")
    ).sort("MovieID").show(3)
# 1010: 16, 1012: 2, 1013: 16

+-------+------------------+
|MovieID|so_nguoi_khac_nhau|
+-------+------------------+
|   1010|                16|
|   1012|                 2|
|   1013|                16|
+-------+------------------+
only showing top 3 rows



### Bước 3. Giải lại Bài 1 bằng DataFrame

14 dòng kết quả phải trùng từng con số với Bước 2 của Lab 2 D.1.

In [21]:
tb = df_ratings.groupBy("MovieID").agg(
        sround(avg("Rating"), 2).alias("diem_tb"),
        count("*").alias("so_luot"))

# join với on="MovieID" cho ra một cột MovieID, không phải tuple lồng nhau như RDD.
ket_qua_df = tb.join(df_movies, on="MovieID") \
               .select("Title", "diem_tb", "so_luot") \
               .orderBy(col("diem_tb").desc(), col("Title"))

ket_qua_df.show(20, truncate=40)
# Dòng đầu : Sunset Boulevard (1950)          4.36   7
# Dòng cuối: The Silence of the Lambs (1991)  3.14   7

+----------------------------------------+-------+-------+
|                                   Title|diem_tb|so_luot|
+----------------------------------------+-------+-------+
|                 Sunset Boulevard (1950)|   4.36|      7|
|                   The Terminator (1984)|   4.06|     18|
|                           Psycho (1960)|    4.0|      2|
|           The Godfather: Part II (1974)|    4.0|     17|
|           No Country for Old Men (2007)|   3.89|     18|
|The Lord of the Rings: The Fellowship...|   3.89|     18|
|               The Social Network (2010)|   3.86|      7|
|The Lord of the Rings: The Return of ...|   3.82|     11|
|       E.T. the Extra-Terrestrial (1982)|   3.67|     18|
|                        Gladiator (2000)|   3.61|     18|
|                       Fight Club (1999)|    3.5|      7|
|               Mad Max: Fury Road (2015)|   3.47|     18|
|               Lawrence of Arabia (1962)|   3.44|     18|
|         The Silence of the Lambs (1991)|   3.14|      

### Bước 4. Tìm phim điểm cao nhất với ngưỡng 5 lượt

In [22]:
cao_nhat = ket_qua_df.filter(col("so_luot") >= 5).first()
print(cao_nhat)
print(cao_nhat["Title"], cao_nhat["diem_tb"])
# Row(Title='Sunset Boulevard (1950)', diem_tb=4.36, so_luot=7)
# Sunset Boulevard (1950) 4.36

# ket_qua_df đã được sắp giảm dần theo điểm nên chỉ cần lọc rồi lấy dòng đầu,
# không cần max(). Kết quả trùng khít với Bước 3 của Lab 2 D.1.

Row(Title='Sunset Boulevard (1950)', diem_tb=4.36, so_luot=7)
Sunset Boulevard (1950) 4.36


### Bước 5. So hai cách giải

Đếm dòng không rỗng, bỏ comment, đọc dữ liệu và lệnh in; tính cả dòng nối tiếp. Với RDD, tách biểu thức `sorted(...)` khỏi vòng lặp in và tính là một dòng.

| Việc phải làm | RDD (Lab 2) | DataFrame |
|---|---|---|
| Khai báo kiểu dữ liệu | hàm `parse_rating` tự viết | chuỗi DDL, khai báo một lần |
| Đọc hai tệp ratings | `union` hai RDD | truyền danh sách đường dẫn |
| Tính trung bình theo khoá | tự mang cặp `(tổng, đếm)` qua `reduceByKey` rồi chia bằng `mapValues` | một lệnh `agg(avg, count)` |
| Lấy tên phim | `join` rồi tự lần theo chỉ số `x[1][1][0]` | `join(on="MovieID")` rồi `select` theo tên cột |
| Sắp xếp | `collect()` về driver rồi `sorted()` của Python | `orderBy()` chạy trong Spark |
| Số dòng của phần tính toán | 6 | 6 |
Số dòng dựa trên Lab 2 D.1 Bước 1–2 sau khi điền phép nối: RDD có 3 dòng gộp, 2 dòng nối/chọn và 1 dòng sắp xếp; bản DataFrame ở Bước 3 có 3 dòng gộp và 3 dòng nối/chọn/sắp xếp.


### Câu hỏi kiểm tra nhanh

1. `count("*")` và `count("Rating")` khác nhau ở điểm nào? Trên bộ dữ liệu này hai
   cách có ra cùng kết quả không?
2. Vì sao `join(df_movies, on="MovieID")` chỉ cho ra một cột `MovieID`, trong khi
   `join` của RDD cho ra một tuple lồng nhau?
3. Bước 3 dùng `orderBy` với hai khoá: điểm giảm dần rồi tên phim tăng dần. Khoá thứ
   hai dùng để làm gì, biết rằng Lab 2 D.1 đã nhắc về hai cặp phim bằng điểm?
**Trả lời:** `count("*")` đếm cả dòng có Rating NULL, còn `count("Rating")` bỏ các NULL; hai cách bằng nhau nếu cột này không có NULL. Nối theo tên cột giữ một khóa chung vì Spark biết schema. Khóa phụ Title giúp thứ tự ổn định khi bằng điểm.


## B.4. Nối dữ liệu với `join`

### Mục tiêu

- Biết cú pháp `join` của DataFrame và các giá trị của tham số `how`.
- Chọn đúng kiểu nối cho câu hỏi đang cần trả lời.

### Khái niệm cần nắm

Cú pháp: `join(other, on=None, how=None)`, trong đó `other` là DataFrame bên phải,
`on` là tên cột dùng để nối, `how` mặc định là `inner`.

| how | Trả về |
|---|---|
| `inner` | Chỉ các dòng có khoá khớp ở cả hai bên. Đây là mặc định |
| `left` | Mọi dòng của bên trái, kèm dòng khớp của bên phải; không khớp thì cột bên phải là NULL |
| `right` | Mọi dòng của bên phải, kèm dòng khớp của bên trái |
| `full` | Mọi dòng của cả hai bên, khớp và không khớp |
| `left_semi` | Các dòng bên trái có khớp ở bên phải, nhưng chỉ giữ cột của bên trái |
| `left_anti` | Các dòng bên trái KHÔNG khớp ở bên phải, chỉ giữ cột của bên trái |
| `cross` | Tích Descartes của hai bên |

### Bước 1. Ba kiểu nối trên cùng một cặp bảng

In [23]:
# tb là bảng điểm trung bình theo phim, tạo ở B.3 Bước 3.
print(df_movies.join(tb, on="MovieID", how="inner").count())       # 14
print(df_movies.join(tb, on="MovieID", how="left").count())        # 50
print(df_movies.join(tb, on="MovieID", how="left_anti").count())   # 36

14
50
36


### Bước 2. `left join` và giá trị NULL

In [24]:
df_movies.join(tb, on="MovieID", how="left") \
         .select("MovieID", "Title", "diem_tb", "so_luot") \
         .sort("MovieID").show(5, truncate=40)
# Năm phim từ 1001 tới 1005 đều có diem_tb và so_luot là NULL

# Spark 3.5 in giá trị rỗng là NULL; các bản cũ hơn in là null. Muốn thay NULL bằng 0
# thì dùng fillna({"so_luot": 0}), nhưng hãy cẩn thận: với cột diem_tb, số 0 nghĩa là
# có người chấm 0 điểm, khác hẳn với không có ai chấm. Đây đúng là vấn đề đã gặp ở
# Lab 2 D.3 với dòng Psycho (1960).

+-------+-------------------------------+-------+-------+
|MovieID|                          Title|diem_tb|so_luot|
+-------+-------------------------------+-------+-------+
|   1001|           The Godfather (1972)|   NULL|   NULL|
|   1002|The Shawshank Redemption (1994)|   NULL|   NULL|
|   1003|        Schindler's List (1993)|   NULL|   NULL|
|   1004|             Raging Bull (1980)|   NULL|   NULL|
|   1005|              Casablanca (1942)|   NULL|   NULL|
+-------+-------------------------------+-------+-------+
only showing top 5 rows



### Bước 3. `left_anti` để tìm phần dữ liệu bị thiếu

In [25]:
df_movies.join(tb, on="MovieID", how="left_anti") \
         .sort("MovieID").show(5, truncate=40)

# left_anti là công cụ kiểm tra chất lượng dữ liệu rất tiện: nó trả lời câu "những
# dòng nào của bảng này không có đối ứng ở bảng kia". Ở bài tập về nhà bạn sẽ cần nó
# để tìm đơn hàng không có dòng chi tiết và đơn hàng không có đánh giá.

+-------+-------------------------------+-----------------------+
|MovieID|                          Title|                 Genres|
+-------+-------------------------------+-----------------------+
|   1001|           The Godfather (1972)|            Crime|Drama|
|   1002|The Shawshank Redemption (1994)|                  Drama|
|   1003|        Schindler's List (1993)|Biography|Drama|History|
|   1004|             Raging Bull (1980)|  Biography|Drama|Sport|
|   1005|              Casablanca (1942)|      Drama|Romance|War|
+-------+-------------------------------+-----------------------+
only showing top 5 rows



### Câu hỏi kiểm tra nhanh

1. `left_semi` và `inner` khác nhau ở điểm nào, khi khoá nối là duy nhất ở cả hai bảng?
2. Vì sao cách dùng `rightOuterJoin` ở Bước 4 của Lab 2 D.4 lại tương ứng với
   `how="right"` ở đây?
**Trả lời:** Khi khóa duy nhất, hai phép nối giữ cùng tập dòng khớp; `left_semi` chỉ giữ cột bên trái, còn `inner` lấy cột cả hai bên. `rightOuterJoin` và `how="right"` đều giữ toàn bộ khóa bên phải và điền phần không khớp bằng giá trị thiếu.


## B.5. Spark SQL

### Mục tiêu

- Đăng ký view và viết lại một truy vấn bằng SQL thuần.
- Đối chiếu bằng máy rằng hai cách viết cho ra cùng kết quả.

### Khái niệm cần nắm

`createOrReplaceTempView` đặt một cái tên cho DataFrame, sau đó `spark.sql("...")`
truy vấn nó như một bảng trong cơ sở dữ liệu. Tên này chỉ sống trong phiên Spark hiện
tại, không ghi gì xuống đĩa. Spark SQL là một trong năm thành phần chính của Spark.

DataFrame API và SQL cùng dùng bộ tối ưu của Spark. Các biểu thức tương đương có thể có kế hoạch tương tự; dùng `explain()` để kiểm tra kế hoạch cụ thể.

### Bước 1. Viết lại Bài 1 bằng SQL thuần


In [26]:
df_ratings.createOrReplaceTempView("ratings")
df_movies.createOrReplaceTempView("movies")

ket_qua_sql = spark.sql("""
    SELECT m.Title,
           ROUND(AVG(r.Rating), 2) AS diem_tb,
           COUNT(*)                AS so_luot
    FROM ratings r
    JOIN movies  m ON r.MovieID = m.MovieID
    GROUP BY m.Title
    ORDER BY diem_tb DESC, m.Title
""")

ket_qua_sql.show(20, truncate=40)      # 14 dòng, trùng với Bước 3 của B.3
print(ket_qua_sql.exceptAll(ket_qua_df).count())      # 0

# Dòng cuối in ra 0, nghĩa là không có dòng nào của bản SQL vắng mặt trong bản
# DataFrame. Đối chiếu bằng máy như vậy chắc chắn hơn nhìn bằng mắt.
# Đối chiếu hai chiều, kể cả số lần xuất hiện của dòng.
assert ket_qua_sql.exceptAll(ket_qua_df).count() == 0
assert ket_qua_df.exceptAll(ket_qua_sql).count() == 0


+----------------------------------------+-------+-------+
|                                   Title|diem_tb|so_luot|
+----------------------------------------+-------+-------+
|                 Sunset Boulevard (1950)|   4.36|      7|
|                   The Terminator (1984)|   4.06|     18|
|                           Psycho (1960)|    4.0|      2|
|           The Godfather: Part II (1974)|    4.0|     17|
|           No Country for Old Men (2007)|   3.89|     18|
|The Lord of the Rings: The Fellowship...|   3.89|     18|
|               The Social Network (2010)|   3.86|      7|
|The Lord of the Rings: The Return of ...|   3.82|     11|
|       E.T. the Extra-Terrestrial (1982)|   3.67|     18|
|                        Gladiator (2000)|   3.61|     18|
|                       Fight Club (1999)|    3.5|      7|
|               Mad Max: Fury Road (2015)|   3.47|     18|
|               Lawrence of Arabia (1962)|   3.44|     18|
|         The Silence of the Lambs (1991)|   3.14|      

### Bước 2. So hai kế hoạch thực thi

Hai kế hoạch in ra dài và khác nhau ở vài chi tiết đặt tên, nhưng các bước chính phải
giống nhau. Tìm ba thứ sau trong cả hai bản in:

- `HashAggregate`: bước gộp theo khoá, có cả avg và count.
- `Exchange hashpartitioning`: bước shuffle, phân vùng lại dữ liệu theo khoá gộp.
- Một dòng chứa `Join`. Vì `movies` chỉ có 50 dòng, Spark thường chọn
  `BroadcastHashJoin`. Nếu bảng lớn hơn ngưỡng mặc định 10 MB thì nó chọn `SortMergeJoin`.

In [27]:
print("===== Kế hoạch của DataFrame API =====")
ket_qua_df.explain()

print("\n===== Kế hoạch của SQL =====")
ket_qua_sql.explain()

# Ở Bước 3 của Lab 2 D.3, bạn phải tự quyết định thay join bằng broadcast. Với
# DataFrame thì Spark tự nhìn kích thước bảng và tự chọn. Lược đồ không chỉ làm code
# ngắn hơn, nó cho Spark đủ thông tin để tối ưu thay bạn.

===== Kế hoạch của DataFrame API =====
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [diem_tb#2093 DESC NULLS LAST, Title#45 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(diem_tb#2093 DESC NULLS LAST, Title#45 ASC NULLS FIRST, 8), ENSURE_REQUIREMENTS, [plan_id=5863]
      +- Project [Title#45, diem_tb#2093, so_luot#2095L]
         +- BroadcastHashJoin [MovieID#51], [MovieID#44], Inner, BuildRight, false
            :- HashAggregate(keys=[MovieID#51], functions=[avg(Rating#52), count(1)])
            :  +- Exchange hashpartitioning(MovieID#51, 8), ENSURE_REQUIREMENTS, [plan_id=5856]
            :     +- HashAggregate(keys=[MovieID#51], functions=[partial_avg(Rating#52), partial_count(1)])
            :        +- Filter isnotnull(MovieID#51)
            :           +- InMemoryTableScan [MovieID#51, Rating#52], [isnotnull(MovieID#51)]
            :                 +- InMemoryRelation [UserID#50, MovieID#51, Rating#52, Timestamp#53L], StorageLevel(disk, memor

### Câu hỏi kiểm tra nhanh

1. Câu SQL ở Bước 1 nhóm theo `m.Title`, còn bản DataFrame nhóm theo `MovieID`. Khi nào
   hai cách này cho ra kết quả khác nhau?
**Trả lời:** Nếu nhiều MovieID có cùng Title, SQL nhóm theo Title sẽ gộp các phim đó, còn DataFrame nhóm theo MovieID vẫn tách riêng. Trên bộ dữ liệu này, đối chiếu hai chiều ở trên kiểm tra trực tiếp sự tương đương.


## B.6. Các hàm Window

Đây là mục trọng tâm thứ hai của buổi học.

### Mục tiêu

- Khai báo được một cửa sổ bằng `Window.partitionBy` và `orderBy`.
- Dùng `row_number` để lấy top-N trong từng nhóm.
- Phân biệt `row_number`, `rank` và `dense_rank` trên dữ liệu có giá trị bằng nhau.

### Khái niệm cần nắm

Hàm Window tính một giá trị cho từng dòng nhưng dựa trên một nhóm dòng xung quanh nó.
Khác với `groupBy` ở chỗ: `groupBy` gộp nhiều dòng thành một dòng, còn Window giữ
nguyên số dòng và thêm cột mới. Spark SQL hỗ trợ ba loại hàm Window: hàm xếp thứ hạng,
hàm phân tích và hàm tổng hợp.

Nhờ Window mới trả lời được câu hỏi dạng "mỗi thể loại, hai phim điểm cao nhất là gì".
`groupBy` một mình không làm được: nó cho bạn điểm cao nhất của từng thể loại, nhưng
không cho biết phim nào đạt điểm đó.

Ba hàm xếp thứ hạng:

| Hàm | Cách đánh số khi có dòng bằng giá trị |
|---|---|
| `row_number()` | Đánh số liên tiếp 1, 2, 3, không quan tâm bằng nhau; thứ tự giữa các dòng bằng nhau là tuỳ ý |
| `rank()` | Các dòng bằng nhau cùng một hạng, rồi để trống các hạng kế tiếp |
| `dense_rank()` | Các dòng bằng nhau cùng một hạng, và không để trống hạng nào |

Hai hàm phân tích hay dùng khác là `lag` và `lead`, cho phép đọc giá trị của dòng
trước hoặc dòng sau trong cùng cửa sổ. Rất tiện khi so sánh doanh thu tháng này với
tháng liền trước.

### Bước 1. Trải phẳng cột Genres bằng `explode`

In [28]:
from pyspark.sql.functions import explode

df_the_loai = df_movies.select(
    "MovieID", explode(split(col("Genres"), "\\|")).alias("Genre"))

print(df_the_loai.count())      # 122

# 122 dòng, khớp đúng với Bước 1 của Lab 2 D.2. explode là phép tương đương của
# flatMap bên RDD: một dòng đầu vào sinh ra nhiều dòng đầu ra. Nhớ chắn dấu gạch dọc
# như đã học ở B.2, nếu không sẽ ra 898 dòng.

122


### Bước 2. Hai phim điểm cao nhất mỗi thể loại

In [29]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

bang = df_the_loai.join(tb, on="MovieID") \
                  .join(df_movies.select("MovieID", "Title"), on="MovieID")

# partitionBy: xếp hạng riêng trong từng thể loại.
# orderBy: điểm giảm dần, tên phim là khoá phụ khi bằng điểm.
w = Window.partitionBy("Genre").orderBy(col("diem_tb").desc(), col("Title"))

top2 = bang.withColumn("hang", row_number().over(w)) \
           .filter(col("hang") <= 2) \
           .select("Genre", "hang", "Title", "diem_tb", "so_luot") \
           .orderBy("Genre", "hang")

top2.show(24, truncate=40)
print(top2.count())             # 20

# Hai mươi dòng chứ không phải 24. Trong 12 thể loại có đánh giá, bốn thể loại
# Family, Film-Noir, Horror và Mystery chỉ có đúng một phim được đánh giá, nên không
# có hạng 2.
#
# Psycho (1960) xuất hiện ở ba thể loại Horror, Mystery và Thriller, mỗi lần với cùng
# điểm 4.0. Đó là cách Window hoạt động: một phim thuộc nhiều phân vùng khác nhau và
# được xếp hạng riêng trong từng phân vùng.

+---------+----+----------------------------------------+-------+-------+
|    Genre|hang|                                   Title|diem_tb|so_luot|
+---------+----+----------------------------------------+-------+-------+
|   Action|   1|                   The Terminator (1984)|   4.06|     18|
|   Action|   2|                        Gladiator (2000)|   3.61|     18|
|Adventure|   1|The Lord of the Rings: The Fellowship...|   3.89|     18|
|Adventure|   2|The Lord of the Rings: The Return of ...|   3.82|     11|
|Biography|   1|               The Social Network (2010)|   3.86|      7|
|Biography|   2|               Lawrence of Arabia (1962)|   3.44|     18|
|    Crime|   1|           The Godfather: Part II (1974)|    4.0|     17|
|    Crime|   2|           No Country for Old Men (2007)|   3.89|     18|
|    Drama|   1|                 Sunset Boulevard (1950)|   4.36|      7|
|    Drama|   2|           The Godfather: Part II (1974)|    4.0|     17|
|   Family|   1|       E.T. the Extra-

### Bước 3. `row_number`, `rank` và `dense_rank` trên cùng một cửa sổ

In [30]:
from pyspark.sql.functions import rank, dense_rank

w2 = Window.orderBy(col("diem_tb").desc())

tb.join(df_movies.select("MovieID", "Title"), on="MovieID") \
  .withColumn("rn", row_number().over(w2)) \
  .withColumn("rk", rank().over(w2)) \
  .withColumn("dr", dense_rank().over(w2)) \
  .select("Title", "diem_tb", "rn", "rk", "dr") \
  .orderBy("rn").show(14, truncate=40)

# Ba hàm chỉ khác nhau khi có dòng bằng giá trị. Psycho và The Godfather: Part II
# cùng 4.0: row_number đánh 3 và 4 một cách tuỳ ý, rank đánh cả hai là 3 rồi nhảy
# sang 5, dense_rank đánh cả hai là 3 rồi tiếp tục 4.
#
# Cửa sổ w2 không có partitionBy, nên Spark coi toàn bộ dữ liệu là một phân vùng duy
# nhất và dồn hết về một executor. Với 14 dòng thì không sao, nhưng trên dữ liệu lớn
# cách này có thể tạo nút thắt bộ nhớ và hiệu năng. Spark cũng in ra cảnh báo WARN WindowExec.
#
# Mẫu ở Bước 2 và Bước 3 chính là mẫu cần dùng cho bài xếp hạng người bán trong bài
# tập về nhà. Khác biệt duy nhất là tên bảng và tên cột, còn khung
# Window.partitionBy(...).orderBy(...) rồi row_number().over(w) thì giữ nguyên.
# Chọn row_number, rank hay dense_rank thì phải giải thích được vì sao.


+----------------------------------------+-------+---+---+---+
|                                   Title|diem_tb| rn| rk| dr|
+----------------------------------------+-------+---+---+---+
|                 Sunset Boulevard (1950)|   4.36|  1|  1|  1|
|                   The Terminator (1984)|   4.06|  2|  2|  2|
|           The Godfather: Part II (1974)|    4.0|  3|  3|  3|
|                           Psycho (1960)|    4.0|  4|  3|  3|
|           No Country for Old Men (2007)|   3.89|  5|  5|  4|
|The Lord of the Rings: The Fellowship...|   3.89|  6|  5|  4|
|               The Social Network (2010)|   3.86|  7|  7|  5|
|The Lord of the Rings: The Return of ...|   3.82|  8|  8|  6|
|       E.T. the Extra-Terrestrial (1982)|   3.67|  9|  9|  7|
|                        Gladiator (2000)|   3.61| 10| 10|  8|
|                       Fight Club (1999)|    3.5| 11| 11|  9|
|               Mad Max: Fury Road (2015)|   3.47| 12| 12| 10|
|               Lawrence of Arabia (1962)|   3.44| 13| 

---
# Dọn dẹp

In [31]:
spark.stop()
print("Đã đóng SparkSession.")

Đã đóng SparkSession.
